# Time-series analysis

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-10-time-series.ipynb)

Time-series SQL comes down to three techniques: bucket timestamps into periods, fill the periods that have no data,
and compare each row with its neighbours in time. Most of the difficulty is at the edges: time zones, boundaries,
and days with no data at all.

This chapter covers period buckets, calendar spines, gaps and islands, sessions, cohorts and cumulative counts. It
ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Set the session time zone once, so timestamps mean the same moment on every machine.

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW events AS
SELECT * FROM VALUES
  (1,  1, TIMESTAMP '2023-01-01 09:00:00', 'login'),
  (2,  1, TIMESTAMP '2023-01-02 09:05:00', 'login'),
  (3,  1, TIMESTAMP '2023-01-03 08:55:00', 'login'),
  (4,  1, TIMESTAMP '2023-01-05 09:00:00', 'login'),
  (5,  1, TIMESTAMP '2023-01-06 09:10:00', 'login'),
  (6,  2, TIMESTAMP '2023-01-01 10:00:00', 'login'),
  (7,  2, TIMESTAMP '2023-01-02 10:00:00', 'login'),
  (8,  2, TIMESTAMP '2023-01-04 10:00:00', 'login'),
  (9,  3, TIMESTAMP '2023-02-01 10:00:00', 'view'),
  (10, 3, TIMESTAMP '2023-02-01 10:05:00', 'click'),
  (11, 3, TIMESTAMP '2023-02-01 10:25:00', 'view'),
  (12, 3, TIMESTAMP '2023-02-01 11:30:00', 'purchase'),
  (13, 3, TIMESTAMP '2023-02-01 11:45:00', 'view'),
  (14, 4, TIMESTAMP '2023-02-01 10:00:00', 'login'),
  (15, 4, TIMESTAMP '2023-02-01 10:20:00', 'view')
AS events(event_id, user_id, event_ts, event_type)
""")

## 1. Time zones and boundaries

Keep timestamps in UTC inside the warehouse, and convert only for display. For time ranges, use a **half-open**
interval `[start, end)`: a row exactly at `end` belongs to the next period, so no row is counted twice and none is
lost.

The `events` table holds timestamps. Its values are UTC.

## 2. Bucketing into periods

`trunc(date, 'MM')` returns the first day of the month. Group by that value to get one row per month. Use `'WEEK'`
or `'QUARTER'` for other periods.

In [ ]:
spark.sql("""
SELECT trunc(order_date, 'MM') AS month_start, COUNT(*) AS orders
FROM orders
GROUP BY trunc(order_date, 'MM')
ORDER BY month_start
""").show()

## 3. Filling gaps with a calendar spine

A day with no logins is absent from a `GROUP BY`, so a chart built from it shows a hole instead of a zero. Build the
calendar first with `sequence` and `explode`, then `LEFT JOIN` the activity onto it. `COALESCE` turns the missing
counts into zeros. Here 2023-01-07 has no logins.

In [ ]:
spark.sql("""
WITH spine AS (
  SELECT explode(sequence(DATE '2023-01-01', DATE '2023-01-07', INTERVAL 1 DAY)) AS day
),
daily AS (
  SELECT CAST(event_ts AS DATE) AS day, COUNT(*) AS logins
  FROM events
  WHERE event_type = 'login'
  GROUP BY CAST(event_ts AS DATE)
)
SELECT s.day, COALESCE(d.logins, 0) AS logins
FROM spine s
LEFT JOIN daily d ON d.day = s.day
ORDER BY s.day
""").show()

## 4. Gaps and islands

Within a run of consecutive days, subtracting the row number from the date gives a constant. That constant names the
run, so grouping by it finds each **island**. Deduplicate to one row per day first, or several events on one day
shift the row numbers.

`date_sub(d, n)` subtracts `n` days, and `n` must be an integer, so the row number is cast.

In [ ]:
spark.sql("""
WITH daily AS (
  SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
),
grouped AS (
  SELECT user_id, d,
         date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
  FROM daily
)
SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS days
FROM grouped
GROUP BY user_id, island
ORDER BY user_id, start_day
""").show()

## 5. Sessions

A new session starts when the gap to the same user's previous event is more than 30 minutes. Use `LAG` to get the
previous timestamp, flag the starts of sessions, then take a running sum of the flags to number them. `unix_timestamp`
converts to seconds, so the gap in minutes is the difference divided by 60.

In [ ]:
spark.sql("""
WITH ordered AS (
  SELECT user_id, event_ts, event_id,
         LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
  FROM events
),
flagged AS (
  SELECT user_id, event_ts, event_id,
         CASE WHEN prev_ts IS NULL
                OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
              THEN 1 ELSE 0 END AS starts_session
  FROM ordered
)
SELECT user_id, event_ts,
       SUM(starts_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_no
FROM flagged
WHERE user_id IN (3, 4)
ORDER BY user_id, event_ts
""").show()

## 6. Cohorts and retention

A **cohort** is the group of customers who first appeared in the same period. For each cohort and each later month,
count the distinct customers who were active. `COUNT(DISTINCT ...)` makes a customer with several orders in one month
count once.

In [ ]:
spark.sql("""
WITH first_order AS (
  SELECT customer_id, MIN(date_format(order_date, 'yyyy-MM')) AS cohort_month
  FROM orders
  GROUP BY customer_id
),
activity AS (
  SELECT DISTINCT customer_id, date_format(order_date, 'yyyy-MM') AS activity_month
  FROM orders
)
SELECT f.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id) AS active_customers
FROM first_order f
JOIN activity a ON a.customer_id = f.customer_id
GROUP BY f.cohort_month, a.activity_month
ORDER BY f.cohort_month, a.activity_month
""").show()

## 7. Cumulative counts

"How many distinct customers had ordered by this date?" A correlated `COUNT(DISTINCT)` answers it, but it recounts the
history for every date. Spark does not allow `DISTINCT` inside a window. Instead, find each customer's first order once,
then take a running sum of new customers per day.

In [ ]:
spark.sql("""
WITH first_order AS (
  SELECT customer_id, MIN(order_date) AS first_date
  FROM orders
  GROUP BY customer_id
)
SELECT first_date,
       COUNT(*) AS new_customers,
       SUM(COUNT(*)) OVER (ORDER BY first_date) AS cumulative_customers
FROM first_order
GROUP BY first_date
ORDER BY first_date
""").show()

## Interview notes

- **Late and out-of-order data.** Events arrive after their period has been reported. Reprocess the affected
  periods, or accept a short window of lateness and say so.
- **Define the session.** "A new session after 30 minutes" is a business rule. Ask whether the gap is measured
  between consecutive events, and whether a session can span midnight.
- **Time zones.** Store timestamps in UTC and convert for display. Daylight saving changes make local-time day
  lengths differ, which breaks naive "24 hours = one day" arithmetic.

## Summary

- Bucket with `trunc`, `date_format` or `date_trunc`, and say which time zone the buckets use.
- Build a calendar spine before reporting, so empty periods show as zero.
- Gaps and islands and sessions both compare each row with the previous one, then accumulate. Learn the pattern once.
- Use half-open intervals for every time range.

## Practice questions

1. How many orders were placed in each month? Sort by month.
2. For the days 2023-01-01 to 2023-01-07, how many logins happened each day? Include zeros. Sort by day.
3. How many sessions does each user have, using the 30-minute rule? Sort by `user_id`.

## Answers

### Question 1

In [ ]:
from datetime import date

q = "SELECT trunc(order_date, 'MM') AS month_start, COUNT(*) AS orders FROM orders GROUP BY trunc(order_date, 'MM') ORDER BY month_start"
spark.sql(q).show()
assert spark.sql(q).collect() == [(date(2023, 1, 1), 4), (date(2023, 2, 1), 5), (date(2023, 3, 1), 5)]

### Question 2

In [ ]:
q = """
WITH spine AS (
  SELECT explode(sequence(DATE '2023-01-01', DATE '2023-01-07', INTERVAL 1 DAY)) AS day
),
daily AS (
  SELECT CAST(event_ts AS DATE) AS day, COUNT(*) AS logins
  FROM events WHERE event_type = 'login'
  GROUP BY CAST(event_ts AS DATE)
)
SELECT s.day, COALESCE(d.logins, 0) AS logins
FROM spine s LEFT JOIN daily d ON d.day = s.day
ORDER BY s.day
"""
spark.sql(q).show()
from datetime import date

assert spark.sql(q).collect() == [
    (date(2023, 1, 1), 2), (date(2023, 1, 2), 2), (date(2023, 1, 3), 1), (date(2023, 1, 4), 1),
    (date(2023, 1, 5), 1), (date(2023, 1, 6), 1), (date(2023, 1, 7), 0),
]

### Question 3

In [ ]:
q = """
WITH ordered AS (
  SELECT user_id, event_ts, event_id,
         LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
  FROM events
),
flagged AS (
  SELECT user_id, event_ts, event_id,
         CASE WHEN prev_ts IS NULL
                OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
              THEN 1 ELSE 0 END AS starts_session
  FROM ordered
)
SELECT user_id, MAX(session_no) AS sessions
FROM (
  SELECT user_id, SUM(starts_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_no
  FROM flagged
)
GROUP BY user_id
ORDER BY user_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 5), (2, 3), (3, 2), (4, 1)]